<a href="https://colab.research.google.com/github/pradh/tools/blob/ml/dc-embed/SEARCH_StatVarEmbeddings_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
BUILDS = ['demographics300', 'uncurated3000']

## Load model, copy over and load embeddings


In [ ]:
%%capture
from google.colab import auth
auth.authenticate_user()

!gcloud config set project 'datcom-204919'
!gsutil -m cp gs://datcom-csv/embeddings/embeddings_*.csv .

In [ ]:
!ls -l

total 18168
-rw-r--r-- 1 root root  2187591 Dec 16 16:06 embeddings_demographics300.csv
-rw-r--r-- 1 root root 16402373 Dec 16 16:06 embeddings_uncurated3000.csv
drwxr-xr-x 2 root root     4096 Dec 16 15:34 flagged
drwxr-xr-x 1 root root     4096 Dec 16 00:01 sample_data


In [ ]:
%%capture
!pip install -U sentence-transformers
!pip install datasets
!pip install -q gradio

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Download model
model = SentenceTransformer('all-MiniLM-L6-v2')

import torch
from datasets import load_dataset

# Load for Search below
dataset_embeddings_maps = {}
dcid_maps = {}
for build in BUILDS:
  print('Loading build ', build)
  ds = load_dataset('csv', data_files=f'embeddings_{build}.csv')

  df = ds["train"].to_pandas()
  dcid_maps[build] = df['dcid'].values.tolist()
  df = df.drop('dcid', axis=1)

  dataset_embeddings_maps[build] = torch.from_numpy(df.to_numpy()).to(torch.float)

Loading build  demographics300


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-0016a5a0ead43565/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

Loading build  uncurated3000


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-0c5a27b6a7fa7813/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

## Fire up a gradio instance


In [ ]:
import gradio as gr
import pandas as pd
from sentence_transformers.util import semantic_search

def inference(build, query):
  query_embeddings = model.encode([query])

  # Note: multiple results may map to the same DCID. As well, the same string may
  hits = semantic_search(query_embeddings, dataset_embeddings_maps[build], top_k=15)
  # map to multiple DCIDs with the same score.
  sv2score = {}
  score2svs = {}
  for e in hits[0]:
    for d in dcid_maps[build][e['corpus_id']].split(','):
      s = e['score']
      # Prefer the top score.
      if d not in sv2score:
        sv2score[d] = s
        if s not in score2svs:
          score2svs[s] = [d]
        else:
          score2svs[s].append(d)

  # Sort by scores
  scores = [s for s in sorted(score2svs.keys(), reverse=True)]
  svs = [' : '.join(score2svs[s]) for s in scores]

  # Addd to Pandas
  result = pd.DataFrame({'SV': svs, 'Cosine Score': scores})
  return result


title = "DC Search Demo"
description = """
Try querying for StatVars.

- "demographics300": 300 SVs with curated descriptions (http://shortn/_iJbtpD2uwF)
  related to demographics
- "uncurated3000": 3000 SVs with only auto-generated name related to
  demographics, crime, agriculture, households, housing, emissions, health
"""

iface = gr.Interface(fn=inference,
                     inputs=[
                         gr.Dropdown(choices=BUILDS,
                                     value='demographics300',
                                     label='Embeddings Build'),
                         gr.Textbox(label='Query',
                                    placeholder='how long do people live?')
                     ],
                     outputs=gr.Dataframe(headers=['SV', 'Cosine Score'],
                                          label='Search Results'),
                     title=title,
                     description=description,
                     flagging="manual",
                     flagging_options=["not at all related",
                                       "related but not ranked right"])

iface.launch(share=True)

/usr/local/lib/python3.8/dist-packages/gradio/deprecation.py:43: UserWarning: You have unused kwarg parameters in Interface, please remove them: {'flagging': 'manual'}
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set `debug=True` in `launch()`

Setting up a public link... we have recently upgraded the way public links are generated. If you encounter any problems, please report the issue and downgrade to gradio version 3.13.0
.
Running on public URL: https://3836313e-8339-4e80.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades (NEW!), check out Spaces: https://huggingface.co/spaces
